In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU & GÁN NHÃN & GỘP
# ==========================================
print("1. Đang tải và gộp dữ liệu...")
df_fake = pd.read_csv('../data/Fake.csv')
df_true = pd.read_csv('../data/True.csv')

# Gán nhãn: 0 cho True (tin thật), 1 cho Fake (tin giả)
df_true['label'] = 0
df_fake['label'] = 1

# Gộp dữ liệu và xáo trộn ngẫu nhiên
df = pd.concat([df_true, df_fake], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Tạo cột 'content' kết hợp tiêu đề và nội dung
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')


# ==========================================
# BƯỚC 2: KIỂM TRA & LÀM SẠCH DỮ LIỆU CƠ BẢN
# ==========================================
print("\n2. Kiểm tra và loại bỏ dữ liệu trùng lặp...")
initial_count = len(df)
df = df.drop_duplicates(subset=['content']).reset_index(drop=True)
print(f"- Đã xóa {initial_count - len(df)} dòng trùng lặp.")
print(f"- Số dòng còn lại: {len(df)}")


# ==========================================
# BƯỚC 3: TIỀN XỬ LÝ VĂN BẢN (TEXT PREPROCESSING)
# ==========================================
print("\n3. Đang tiến hành tiền xử lý văn bản (có thể mất 1-2 phút)...")
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Chuyển về chữ thường
    text = text.lower()
    # Xóa URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Xóa thẻ HTML
    text = re.sub(r'<.*?>', '', text)
    # Xóa ký tự đặc biệt, dấu câu và chữ số
    text = re.sub(r'[^a-z\s]', '', text)
    # Tách từ, lọc từ dừng (stopwords) và loại bỏ từ quá ngắn (<= 1 ký tự)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 1]
    return ' '.join(words)

df['clean_content'] = df['content'].apply(clean_text)
print("- Tiền xử lý văn bản hoàn tất!")


# ==========================================
# BƯỚC 4: TRÍCH XUẤT ĐẶC TRƯNG (TF-IDF)
# ==========================================
print("\n4. Trích xuất đặc trưng bằng TF-IDF Vectorizer...")
# Lấy 5,000 từ xuất hiện phổ biến và mang tính phân loại cao nhất
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_content'])
y = df['label'].values

print(f"- Kích thước ma trận đặc trưng X: {X.shape}")
print(f"- Kích thước mảng nhãn y: {y.shape}")


# ==========================================
# BƯỚC 5: CHIA TẬP TRAIN / TEST
# ==========================================
print("\n5. Chia tập dữ liệu Train / Test...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"- Số lượng mẫu huấn luyện (Train): {X_train.shape[0]}")
print(f"- Số lượng mẫu kiểm thử (Test): {X_test.shape[0]}")
print("\n>>> HOÀN THÀNH TOÀN BỘ TIỀN XỬ LÝ DỮ LIỆU! <<<")

1. Đang tải và gộp dữ liệu...

2. Kiểm tra và loại bỏ dữ liệu trùng lặp...
- Đã xóa 5793 dòng trùng lặp.
- Số dòng còn lại: 39105

3. Đang tiến hành tiền xử lý văn bản (có thể mất 1-2 phút)...
- Tiền xử lý văn bản hoàn tất!

4. Trích xuất đặc trưng bằng TF-IDF Vectorizer...
- Kích thước ma trận đặc trưng X: (39105, 5000)
- Kích thước mảng nhãn y: (39105,)

5. Chia tập dữ liệu Train / Test...
- Số lượng mẫu huấn luyện (Train): 31284
- Số lượng mẫu kiểm thử (Test): 7821

>>> HOÀN THÀNH TOÀN BỘ TIỀN XỬ LÝ DỮ LIỆU! <<<


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Khởi tạo và huấn luyện mô hình Logistic Regression
print("1. Đang huấn luyện mô hình Logistic Regression...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("- Huấn luyện hoàn tất!")

# 2. Dự đoán trên tập kiểm thử (Test)
y_pred = model.predict(X_test)

# 3. Đánh giá độ chính xác
acc = accuracy_score(y_test, y_pred)
print(f"\n2. Độ chính xác (Accuracy): {acc * 100:.2f}%\n")

# 4. Hiển thị báo cáo chi tiết (Precision, Recall, F1-score)
print("--- BÁO CÁO CHI TIẾT (CLASSIFICATION REPORT) ---")
print(classification_report(y_test, y_pred, target_names=['True News (0)', 'Fake News (1)']))

# 5. Vẽ Confusion Matrix (Ma trận nhầm lẫn)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['True (0)', 'Fake (1)'], 
            yticklabels=['True (0)', 'Fake (1)'])
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.title('Confusion Matrix')
plt.show()

ModuleNotFoundError: No module named 'seaborn'